# Tiny-LLM — Google Colab GPU training

Run **Runtime → Change runtime type → T4 GPU** (if offered), then execute cells top to bottom. GPU access is not guaranteed on free Colab. The notebook trains the actual repository's `train.py`, with CUDA enabled by a small companion change in this pull request. No MariaDB credentials are needed.

In [ ]:
import torch
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('No GPU assigned; training will use CPU.')

## Clone repository and install dependencies

Colab normally already includes PyTorch. Install NumPy only if needed; **do not** install the repository's fully pinned `requirements.txt` here, since its CUDA/PyTorch pins may conflict with Colab's runtime.

In [ ]:
import os, subprocess, sys
from pathlib import Path
repo = Path('/content/tiny-llm')
if not repo.exists():
    subprocess.run(['git', 'clone', 'https://github.com/oppiet30/tiny-llm.git', str(repo)], check=True)
os.chdir(repo)
import numpy
print('Repository:', repo)
print('NumPy:', numpy.__version__)

## Dataset

Use the repository's `data/input.txt` if present. Otherwise upload a UTF-8 `.txt` corpus in the next cell. Use text you have permission to train on. The sample needs enough characters for both training and validation (at least ~1,300 characters).

In [ ]:
from google.colab import files
source = Path('data/input.txt')
if not source.exists():
    print('Upload a UTF-8 training text file:')
    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError('No text file uploaded.')
    filename = next(iter(uploaded))
    source.parent.mkdir(parents=True, exist_ok=True)
    source.write_bytes(Path(filename).read_bytes())
print('Corpus:', source, 'bytes:', source.stat().st_size)
subprocess.run([sys.executable, 'prepare_data.py', '--input', str(source)], check=True)

## Train and save checkpoints

The default is a short **250-step** trial. Increase `steps` after confirming it works. Checkpoints are written to Google Drive to survive Colab disconnects. Mounting Drive asks for your permission. The `--resume` option loads a prior checkpoint; only load checkpoints you trust.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
checkpoint_dir = Path('/content/drive/MyDrive/tiny-llm-checkpoints')
checkpoint_dir.mkdir(parents=True, exist_ok=True)
steps = 250  # Increase after the trial run
interval = 100
latest = checkpoint_dir / 'model.pt'
command = [sys.executable, 'train.py', '--steps', str(steps), '--checkpoint-interval', str(interval), '--checkpoint-dir', str(checkpoint_dir)]
if latest.exists():
    print('Existing checkpoint found:', latest)
    # Resume only if the saved step is below the requested target.
    info = checkpoint_dir / 'checkpoint.txt'
    saved_step = int(dict(line.strip().split('=', 1) for line in info.read_text().splitlines())['step']) if info.exists() else None
    if saved_step is not None and saved_step < steps:
        command += ['--resume', str(latest)]
    elif saved_step is not None:
        print('Already at step', saved_step, '; raise steps to continue training.')
        command = None
if command:
    print('Running:', ' '.join(command))
    subprocess.run(command, check=True)

## Results

Your latest model and step-numbered checkpoints are in `MyDrive/tiny-llm-checkpoints`. Training also writes benchmark SQL under the temporary Colab repository's `benchmarks/` directory; this notebook does **not** upload to MariaDB. Copy any SQL you want to keep before the runtime ends.

In [ ]:
print('Checkpoints:')
for p in sorted(checkpoint_dir.glob('*.pt')):
    print(p.name, f'{p.stat().st_size/1024/1024:.1f} MiB')
print('Benchmark SQL:')
for p in sorted(Path('benchmarks').glob('*.sql')):
    print(p)